# LeetCode 1911 — Maximum Alternating Subsequence Sum bằng Python

Link đề: https://leetcode.com/problems/maximum-alternating-subsequence-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def maxAlternatingSum(self, nums: List[int]) -> int:
        ...
```

Lộ trình: **vét cạn mọi dãy con** `O(2ⁿ · n)` → **DP "kết thúc tại i" `O(n²)`** → **DP hai trạng thái chẵn/lẻ** `O(n)` / `O(1)` (lời giải nộp) → **tham lam "cộng mọi bước tăng"** (giống bài mua bán cổ phiếu II).

## 1. Mục tiêu học

1. Phân biệt **dãy con** (subsequence, giữ thứ tự, không cần liên tiếp) với **mảng con** (subarray, liên tiếp).
2. Thiết kế DP theo **trạng thái "số phần tử đã chọn chẵn hay lẻ"** — dấu của phần tử kế tiếp phụ thuộc trạng thái đó.
3. Rút gọn DP về **hai biến** cập nhật song song, bộ nhớ `O(1)`.
4. Nhận ra lời giải tham lam: đáp án = `nums[0] + Σ max(0, nums[i] − nums[i−1])` — và chứng minh nó bằng cách nhìn dãy như **các đoạn tăng**.
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed; tránh lỗi trả sai trạng thái, đệ quy sâu.

## 2. Đề bài & ví dụ

**Tổng xen kẽ** của một mảng (đánh số từ 0) = tổng các phần tử ở chỉ số **chẵn** trừ tổng các phần tử ở chỉ số **lẻ**. Ví dụ tổng xen kẽ của `[4,2,5,3]` là `(4 + 5) − (2 + 3) = 4`.

Cho mảng `nums`, trả về **tổng xen kẽ lớn nhất** của một **dãy con** bất kỳ của `nums` (sau khi đánh số lại các phần tử của dãy con từ 0).

Dãy con: xóa một số phần tử (có thể không xóa) mà không đổi thứ tự các phần tử còn lại. Ví dụ `[2,7,4]` là dãy con của `[4,2,3,7,2,1,4]`, còn `[2,4,2]` thì không.

**Ví dụ 1:** `[4,2,5,3]` → `7` (dãy con `[4,2,5]`: `4 − 2 + 5`).

**Ví dụ 2:** `[5,6,7,8]` → `8` (dãy con `[8]`).

**Ví dụ 3:** `[6,2,1,2,4,5]` → `10` (dãy con `[6,1,5]`: `6 − 1 + 5`).

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `1 ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import combinations
from math import inf
from typing import List

examples = [([4, 2, 5, 3], 7), ([5, 6, 7, 8], 8), ([6, 2, 1, 2, 4, 5], 10)]
for nums, want in examples:
    print(nums, "-> kỳ vọng", want)

[4, 2, 5, 3] -> kỳ vọng 7
[5, 6, 7, 8] -> kỳ vọng 8
[6, 2, 1, 2, 4, 5] -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1 Tổng xen kẽ của một dãy

In [2]:
def alt_sum(seq):
    return sum(x if i % 2 == 0 else -x for i, x in enumerate(seq))

print(alt_sum([4, 2, 5, 3]), alt_sum([4, 2, 5]), alt_sum([6, 1, 5]), alt_sum([]))

4 7 10 0


### 3.2 Liệt kê dãy con bằng `itertools.combinations`

`combinations(range(n), k)` cho mọi bộ `k` chỉ số **tăng dần** — đúng là các dãy con độ dài `k`. Tổng cộng `2ⁿ` dãy con.

In [3]:
nums = [4, 2, 5]
for k in range(len(nums) + 1):
    for idx in combinations(range(len(nums)), k):
        seq = [nums[i] for i in idx]
        print(idx, seq, "->", alt_sum(seq))

() [] -> 0
(0,) [4] -> 4
(1,) [2] -> 2
(2,) [5] -> 5
(0, 1) [4, 2] -> 2
(0, 2) [4, 5] -> -1
(1, 2) [2, 5] -> -3
(0, 1, 2) [4, 2, 5] -> 7


### 3.3 Gán song song

`a, b = f(a, b), g(a, b)` tính cả hai vế phải bằng giá trị **cũ** rồi mới gán — quan trọng khi hai trạng thái DP phụ thuộc lẫn nhau.

In [4]:
even, odd = 0, -inf
x = 4
even, odd = max(even, odd - x), max(odd, even + x)
print(even, odd)

0 4


### 3.4 `math.inf` cho trạng thái chưa đạt được

`-inf` biểu thị "chưa có dãy con nào có số phần tử lẻ"; `-inf - x == -inf`, `max(-inf, v) == v`.

In [5]:
print(-inf - 5, max(-inf, 3))

-inf 3


### 3.5 Tổng các "bước tăng"

`Σ max(0, a[i] − a[i−1])` cộng mọi lần dãy **đi lên**. Đây là công thức của LeetCode 122 (*Best Time to Buy and Sell Stock II*).

In [6]:
a = [6, 2, 1, 2, 4, 5]
ups = [max(0, a[i] - a[i - 1]) for i in range(1, len(a))]
print("các bước tăng:", ups, "-> tổng", sum(ups))

các bước tăng: [0, 0, 1, 2, 1] -> tổng 4


### 3.6 Máy trạng thái hai nút

Có thể vẽ DP như một máy hai trạng thái; mỗi phần tử `x` cho phép **đứng yên** (bỏ `x`) hoặc **chuyển** sang trạng thái kia (chọn `x`):

```
          chọn x (+x)
   EVEN ──────────────▶ ODD
    ▲                    │
    └────────────────────┘
          chọn x (−x)
```

Ta luôn giữ giá trị tốt nhất ở mỗi nút. Kiểu suy nghĩ này dùng lại cho mọi bài "trạng thái luân phiên": cầm/không cầm cổ phiếu, đang trong/ngoài một đoạn, …

## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 1 `[4,2,5,3]`: phần tử ở vị trí chẵn của dãy con được **cộng**, vị trí lẻ bị **trừ**. Muốn lớn: cộng số **lớn**, trừ số **nhỏ**, và luôn kết thúc bằng một phép **cộng** (trừ thêm một số dương ở cuối chỉ làm giảm).

| dãy con | tổng xen kẽ |
|---|---|
| `[5]` | 5 |
| `[4,2,5]` | 4 − 2 + 5 = **7** |
| `[4,3]`… | nhỏ hơn |
| `[4,2,5,3]` | 4 |

Quan sát Ví dụ 3 `[6,2,1,2,4,5]`: chọn `6` (đỉnh), trừ `1` (đáy), cộng `5` (đỉnh sau). Mỗi cặp "trừ đáy, cộng đỉnh" có lợi đúng bằng **độ cao của đoạn tăng** từ đáy lên đỉnh. Đây là gốc của lời giải tham lam.

## 5. Cách 1 — Brute Force / baseline: mọi dãy con

In [7]:
def max_alt_brute(nums):
    n = len(nums)
    best = 0
    for k in range(1, n + 1):
        for idx in combinations(range(n), k):
            best = max(best, alt_sum([nums[i] for i in idx]))
    return best

for nums, want in examples:
    print(max_alt_brute(nums), "| kỳ vọng", want)

7 | kỳ vọng 7
8 | kỳ vọng 8
10 | kỳ vọng 10


**Độ phức tạp:** `2ⁿ` dãy con × `O(n)` ⇒ `O(2ⁿ · n)`. `n = 20` đã ~2·10⁷. **Nút thắt:** hai dãy con có cùng "trạng thái cuối" (vị trí cuối, số phần tử chẵn/lẻ) thì phần tiếp theo như nhau ⇒ chỉ cần giữ giá trị **tốt nhất** cho mỗi trạng thái ⇒ DP.

In [8]:
rng = random.Random(0)
arr = [rng.randint(1, 50) for _ in range(16)]
t = time.perf_counter(); r = max_alt_brute(arr); dt = time.perf_counter() - t
print(f"n=16: {r} trong {dt:.2f}s; mỗi phần tử thêm nhân đôi thời gian")

n=16: 134 trong 0.05s; mỗi phần tử thêm nhân đôi thời gian


## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái:** sau khi xét tiền tố `nums[:i]`, giữ
- `even` = tổng xen kẽ lớn nhất của dãy con đã chọn có **số phần tử chẵn** (phần tử kế tiếp sẽ được **cộng**); dãy rỗng cho `even = 0`;
- `odd` = tổng lớn nhất với **số phần tử lẻ** (phần tử kế tiếp sẽ bị **trừ**); ban đầu `−∞`.

**Chuyển với phần tử `x`:**
- *Nếu* chọn `x` từ trạng thái chẵn *thì* `x` được cộng ⇒ ứng viên `odd = even + x`.
- *Nếu* chọn `x` từ trạng thái lẻ *thì* `x` bị trừ ⇒ ứng viên `even = odd − x`.
- *Nếu* bỏ `x` *thì* giữ nguyên.

`even, odd = max(even, odd − x), max(odd, even + x)` (gán song song). **Đáp án:** `odd` (tối ưu luôn kết thúc bằng phép cộng; và `odd ≥ max(nums) > 0 = even ban đầu`).

**Invariant:** sau bước `i`, `even`/`odd` là tối ưu trên mọi dãy con của `nums[:i+1]` theo từng tính chẵn lẻ.

**Insight tham lam:** đáp án = `nums[0] + Σ_{i≥1} max(0, nums[i] − nums[i−1])` — "mua ở đáy, bán ở đỉnh" với giá khởi đầu 0.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP "kết thúc tại `i`" (`O(n²)`)

`E[i]` = tốt nhất của dãy con **kết thúc tại `i`** với `nums[i]` được **cộng**; `O[i]` = kết thúc tại `i` với `nums[i]` bị **trừ**:
- `E[i] = nums[i] + max(0, max_{j<i} O[j])` (bắt đầu mới hoặc nối sau một phần tử bị trừ);
- `O[i] = max_{j<i} E[j] − nums[i]`.

In [9]:
def max_alt_quadratic(nums):
    n = len(nums)
    E = [0] * n
    O = [-inf] * n
    for i in range(n):
        best_o = max([O[j] for j in range(i)], default=-inf)
        best_e = max([E[j] for j in range(i)], default=-inf)
        E[i] = nums[i] + max(0, best_o)
        O[i] = best_e - nums[i]
    return max(E)

for nums, want in examples:
    print(max_alt_quadratic(nums), want)

7 7
8 8
10 10


**Độ phức tạp:** `O(n²)`. Nút thắt: `max_{j<i}` tính lại từ đầu — mà thực ra chỉ cần **giữ max chạy** ⇒ dẫn thẳng tới Cách 3.

### 7.2 Cách 3 — DP hai trạng thái (`O(n)`, `O(1)`) — lời giải chuẩn

In [10]:
def max_alt_dp_demo(nums):
    even, odd = 0, -inf
    print(f"{'x':>4} | {'even':>6} | {'odd':>6}")
    for x in nums:
        even, odd = max(even, odd - x), max(odd, even + x)
        print(f"{x:>4} | {even:>6} | {odd:>6}")
    return odd

print("=>", max_alt_dp_demo([6, 2, 1, 2, 4, 5]))

   x |   even |    odd
   6 |      0 |      6
   2 |      4 |      6
   1 |      5 |      6
   2 |      5 |      7
   4 |      5 |      9
   5 |      5 |     10
=> 10


In [11]:
class SolutionDP:
    def maxAlternatingSum(self, nums: List[int]) -> int:
        even, odd = 0, -inf
        for x in nums:
            even, odd = max(even, odd - x), max(odd, even + x)
        return odd

for nums, want in examples:
    print(SolutionDP().maxAlternatingSum(nums), want)

7 7
8 8
10 10


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Tham lam: cộng mọi bước tăng

In [12]:
class SolutionGreedy:
    def maxAlternatingSum(self, nums: List[int]) -> int:
        ans = nums[0]
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                ans += nums[i] - nums[i - 1]
        return ans

for nums, want in examples:
    print(SolutionGreedy().maxAlternatingSum(nums), want)

7 7
8 8
10 10


**Độ phức tạp:** `O(n)`, `O(1)`. Cách hiểu: thêm phần tử ảo `0` ở đầu; mỗi đoạn tăng từ đáy `b` lên đỉnh `t` đóng góp `t − b` — tương ứng chọn "`… − b + t`" trong dãy con (với đoạn đầu tiên, đáy là `0` ảo nên chỉ cộng `t`).

### 7.4 Cách 5 — Đệ quy có ghi nhớ (top-down) — để thấy giới hạn đệ quy

In [13]:
def max_alt_memo(nums):
    n = len(nums)
    @lru_cache(maxsize=None)
    def f(i, sign):                    # sign = +1: phần tử chọn tiếp theo được cộng
        if i == n:
            return 0
        return max(f(i + 1, sign), sign * nums[i] + f(i + 1, -sign))
    res = f(0, 1)
    f.cache_clear()
    return res

for nums, want in examples:
    print(max_alt_memo(nums), want)

7 7
8 8
10 10


Đúng và dễ viết, nhưng độ sâu đệ quy bằng `n` — vượt giới hạn 1000 với `n = 10⁵` (Lỗi 5). Chuyển sang vòng lặp (Cách 3) là chuẩn.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 3 `[6,2,1,2,4,5]`:

| x | `odd − x` | `even + x` | `even` mới | `odd` mới |
|---|---|---|---|---|
| — | | | 0 | −∞ |
| 6 | −∞ | 6 | 0 | 6 |
| 2 | 4 | 2 | 4 | 6 |
| 1 | 5 | 5 | 5 | 6 |
| 2 | 4 | 7 | 5 | 7 |
| 4 | 3 | 9 | 5 | 9 |
| 5 | 2 | 10 | 5 | **10** |

`odd = 10` ứng với `6 − 1 + 5` (hoặc `6 − 1 + … `). Trace tham lam: `6 + (2−1) + (4−2) + (5−4) = 6 + 1 + 2 + 1 = 10`. ✔

In [14]:
def trace_dp(nums):
    even, odd = 0, -inf
    for x in nums:
        c1, c2 = odd - x, even + x
        even, odd = max(even, c1), max(odd, c2)
        print(f"x={x}: odd-x={c1}, even+x={c2} -> even={even}, odd={odd}")
    return odd

def trace_greedy(nums):
    ans = nums[0]; print(f"bắt đầu ans = nums[0] = {ans}")
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        if d > 0:
            ans += d; print(f"  {nums[i-1]} -> {nums[i]}: tăng {d}, ans = {ans}")
        else:
            print(f"  {nums[i-1]} -> {nums[i]}: không tăng")
    return ans

trace_dp([6, 2, 1, 2, 4, 5]); print(); trace_greedy([6, 2, 1, 2, 4, 5])

x=6: odd-x=-inf, even+x=6 -> even=0, odd=6
x=2: odd-x=4, even+x=2 -> even=4, odd=6
x=1: odd-x=5, even+x=5 -> even=5, odd=6
x=2: odd-x=4, even+x=7 -> even=5, odd=7
x=4: odd-x=3, even+x=9 -> even=5, odd=9
x=5: odd-x=4, even+x=10 -> even=5, odd=10

bắt đầu ans = nums[0] = 6
  6 -> 2: không tăng
  2 -> 1: không tăng
  1 -> 2: tăng 1, ans = 7
  2 -> 4: tăng 2, ans = 9
  4 -> 5: tăng 1, ans = 10


10

Truy vết **chính dãy con tối ưu** bằng tham lam: chọn các **đỉnh** (cuối mỗi đoạn tăng) và **đáy** (đầu mỗi đoạn tăng, trừ đoạn đầu tiên):

In [15]:
def best_subsequence(nums):
    seq = []
    n = len(nums)
    i = 0
    # đoạn đầu: đi lên tới đỉnh đầu tiên
    while i + 1 < n and nums[i + 1] >= nums[i]:
        i += 1
    seq.append(nums[i])
    while i + 1 < n:
        while i + 1 < n and nums[i + 1] <= nums[i]:     # đi xuống tới đáy
            i += 1
        bottom = nums[i]
        while i + 1 < n and nums[i + 1] >= nums[i]:     # đi lên tới đỉnh
            i += 1
        if nums[i] > bottom:
            seq += [bottom, nums[i]]
    return seq

for nums, want in examples:
    s = best_subsequence(nums)
    print(nums, "-> dãy con", s, "tổng xen kẽ", alt_sum(s), "| kỳ vọng", want)

[4, 2, 5, 3] -> dãy con [4, 2, 5] tổng xen kẽ 7 | kỳ vọng 7
[5, 6, 7, 8] -> dãy con [8] tổng xen kẽ 8 | kỳ vọng 8
[6, 2, 1, 2, 4, 5] -> dãy con [6, 1, 5] tổng xen kẽ 10 | kỳ vọng 10


## 9. Code nộp LeetCode

In [16]:
from math import inf
from typing import List

class Solution:
    def maxAlternatingSum(self, nums: List[int]) -> int:
        even, odd = 0, -inf          # even: đã chọn chẵn phần tử (kế tiếp +), odd: lẻ (kế tiếp −)
        for x in nums:
            even, odd = max(even, odd - x), max(odd, even + x)
        return odd

for nums, want in examples:
    assert Solution().maxAlternatingSum(nums) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**DP (Cách 3).** Gọi `E_i`, `O_i` là giá trị tối ưu trên các dãy con của `nums[:i]` có số phần tử chẵn / lẻ. Một dãy con của `nums[:i+1]` hoặc **không** dùng `nums[i]` (thuộc tập của `nums[:i]`), hoặc dùng `nums[i]` làm phần tử **cuối**:
- nếu số phần tử trở thành lẻ thì trước đó chẵn và `nums[i]` ở vị trí chẵn ⇒ được cộng: `E_i + nums[i]`;
- nếu thành chẵn thì trước đó lẻ và `nums[i]` bị trừ: `O_i − nums[i]`.

⇒ `E_{i+1} = max(E_i, O_i − x)`, `O_{i+1} = max(O_i, E_i + x)` — đúng công thức (cần gán song song để dùng `E_i`, `O_i` cũ). Đáp án là `max(E_n, O_n)`; vì mọi số dương, một dãy con chẵn phần tử `[…, a, b]` có tổng nhỏ hơn dãy bỏ `b` (lẻ phần tử) ⇒ `O_n ≥ E_n`, nên trả `odd`.

**Tham lam (Cách 4).** Thêm `nums[−1] = 0`. *Cận trên:* với dãy con `a₀ − a₁ + a₂ − … + a_{2k}` (chỉ số tăng), viết thành `(a₀ − 0) + Σ_t (a_{2t} − a_{2t−1})`; mỗi số hạng `a_{2t} − a_{2t−1}` (với `a_{2t−1}` đứng trước `a_{2t}` trong mảng) ≤ tổng các bước tăng trên đoạn giữa hai vị trí đó; các đoạn rời nhau ⇒ tổng ≤ `Σ max(0, nums[i] − nums[i−1])` (tính cả bước từ `0` ảo lên `nums[0]`). *Đạt được:* chọn đáy/đỉnh của các đoạn tăng như hàm `best_subsequence`.

Kiểm chứng định nghĩa `E_i`, `O_i` bằng máy: với mọi tiền tố, so `even`/`odd` của DP với giá trị tốt nhất **theo định nghĩa** (vét cạn dãy con của tiền tố, tách theo tính chẵn lẻ số phần tử), và kiểm `odd ≥ even` ở cuối.

In [17]:
def best_by_parity(prefix):
    best = {0: 0, 1: -inf}                      # dãy rỗng: chẵn, tổng 0
    n = len(prefix)
    for k in range(1, n + 1):
        for idx in combinations(range(n), k):
            v = alt_sum([prefix[i] for i in idx])
            best[k % 2] = max(best[k % 2], v)
    return best[0], best[1]

rng_chk = random.Random(17)
for _ in range(300):
    nums = [rng_chk.randint(1, 15) for _ in range(rng_chk.randint(1, 9))]
    even, odd = 0, -inf
    for i, x in enumerate(nums):
        even, odd = max(even, odd - x), max(odd, even + x)
        assert (even, odd) == best_by_parity(nums[:i + 1])
    assert odd >= even
print("even/odd của DP khớp định nghĩa trên mọi tiền tố; luôn có odd ≥ even ở cuối")

even/odd của DP khớp định nghĩa trên mọi tiền tố; luôn có odd ≥ even ở cuối


## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Mọi dãy con | `O(2ⁿ · n)` | `O(n)` |
| 2. DP kết thúc tại `i` | `O(n²)` | `O(n)` |
| 3. DP hai trạng thái | `O(n)` | `O(1)` |
| 4. Tham lam | `O(n)` | `O(1)` |
| 5. Đệ quy ghi nhớ | `O(n)` | `O(n)` (+ stack sâu `n`) |

Giá trị lớn nhất: `10⁵` phần tử × `10⁵` ⇒ tối đa ~`10¹⁰` — vượt `int32`, Python không lo; C++/Java cần `long long`.

In [18]:
rng = random.Random(1911)
big = [rng.randint(1, 10**5) for _ in range(10**5)]
saw = [10**5 if i % 2 == 0 else 1 for i in range(10**5)]     # răng cưa: đáp án ~ 5·10^9
for label, data in [("ngẫu nhiên", big), ("răng cưa", saw)]:
    res = []
    for name, sol in [("DP", SolutionDP()), ("Tham lam", SolutionGreedy()), ("Nộp", Solution())]:
        t = time.perf_counter(); r = sol.maxAlternatingSum(data); res.append(r)
        print(f"[{label}] {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1
t = time.perf_counter(); max_alt_quadratic(big[:2000]); dt = time.perf_counter() - t
print(f"Cách 2 với n=2000: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * 2500:.0f}s")

[ngẫu nhiên] DP: 1669837678 trong 6 ms


[ngẫu nhiên] Tham lam: 1669837678 trong 6 ms
[ngẫu nhiên] Nộp: 1669837678 trong 6 ms
[răng cưa] DP: 4999950001 trong 6 ms


[răng cưa] Tham lam: 4999950001 trong 6 ms
[răng cưa] Nộp: 4999950001 trong 6 ms


Cách 2 với n=2000: 0.10s -> n=10^5 ước tính ≈ 238s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [19]:
sols = [Solution(), SolutionDP(), SolutionGreedy()]
edge = examples + [
    ([1], 1),
    ([7, 7, 7], 7),
    ([1, 2, 3, 4], 4),           # tăng dần: lấy phần tử cuối
    ([4, 3, 2, 1], 4),           # giảm dần: lấy phần tử đầu
    ([3, 1, 3], 5),
    ([5, 1, 5, 1, 5], 13),       # răng cưa
    ([100000] * 3, 100000),
]
for nums, want in edge:
    assert max_alt_brute(nums) == want
    for s in sols:
        assert s.maxAlternatingSum(nums) == want, (nums, s)
    assert max_alt_quadratic(nums) == want == max_alt_memo(nums)
    print("OK", nums, "->", want)

OK [4, 2, 5, 3] -> 7
OK [5, 6, 7, 8] -> 8
OK [6, 2, 1, 2, 4, 5] -> 10
OK [1] -> 1
OK [7, 7, 7] -> 7
OK [1, 2, 3, 4] -> 4
OK [4, 3, 2, 1] -> 4
OK [3, 1, 3] -> 5
OK [5, 1, 5, 1, 5] -> 13
OK [100000, 100000, 100000] -> 100000


### 12.2 Ngẫu nhiên có seed: so với vét cạn

In [20]:
rng = random.Random(42)
for _ in range(1500):
    n = rng.randint(1, 11)
    hi = rng.choice([3, 10, 10**5])
    nums = [rng.randint(1, hi) for _ in range(n)]
    want = max_alt_brute(nums)
    for s in sols:
        assert s.maxAlternatingSum(nums) == want, nums
    assert max_alt_quadratic(nums) == want and max_alt_memo(nums) == want
    assert alt_sum(best_subsequence(nums)) == want
print("1500 case ngẫu nhiên khớp vét cạn (kể cả dãy con truy vết được)")

1500 case ngẫu nhiên khớp vét cạn (kể cả dãy con truy vết được)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Hiểu nhầm thành **mảng con liên tiếp**

In [21]:
def wrong_subarray(nums):
    best = -inf
    for l in range(len(nums)):
        for r in range(l, len(nums)):
            best = max(best, alt_sum(nums[l:r + 1]))   # SAI: chỉ xét đoạn liên tiếp
    return best

for nums, want in examples:
    print(nums, "sai:", wrong_subarray(nums), "| đúng:", want)

[4, 2, 5, 3] sai: 7 | đúng: 7
[5, 6, 7, 8] sai: 8 | đúng: 8
[6, 2, 1, 2, 4, 5] sai: 7 | đúng: 10


**Vì sao sai:** Ví dụ 3 cần dãy con `[6, 1, 5]` — bỏ qua `2` giữa `6` và `1`, bỏ `2, 4` giữa `1` và `5`. Mảng con liên tiếp không làm được. (Đây lại là một biến thể hợp lệ khác — Bài tập 5.)

### Lỗi 2 — Trả về trạng thái `even` (kết thúc bằng phép trừ)

In [22]:
def wrong_return_even(nums):
    even, odd = 0, -inf
    for x in nums:
        even, odd = max(even, odd - x), max(odd, even + x)
    return even                                         # SAI

for nums, want in examples + [([5], 5)]:
    print(nums, "sai:", wrong_return_even(nums), "| đúng:", Solution().maxAlternatingSum(nums))

[4, 2, 5, 3] sai: 4 | đúng: 7
[5, 6, 7, 8] sai: 0 | đúng: 8
[6, 2, 1, 2, 4, 5] sai: 5 | đúng: 10
[5] sai: 0 | đúng: 5


**Vì sao sai:** `even` là tốt nhất với **chẵn** phần tử — luôn kết thúc bằng một phép trừ, nên nhỏ hơn. Với `[5]`, `even = 0`. **Sửa:** trả `odd` (hoặc `max(even, odd)`).

### Lỗi 3 — Tham lam "lấy max, rồi min, rồi max…" toàn cục (bỏ qua thứ tự)

In [23]:
def wrong_global(nums):
    s = sorted(nums, reverse=True)
    return s[0]                                          # SAI kiểu "lấy số lớn nhất"
def wrong_global2(nums):
    return max(nums) - min(nums) + max(nums)             # SAI: dùng một số hai lần, sai thứ tự

for nums, want in examples:
    print(nums, "chỉ max:", wrong_global(nums), "| max−min+max:", wrong_global2(nums), "| đúng:", want)

[4, 2, 5, 3] chỉ max: 5 | max−min+max: 8 | đúng: 7
[5, 6, 7, 8] chỉ max: 8 | max−min+max: 11 | đúng: 8
[6, 2, 1, 2, 4, 5] chỉ max: 6 | max−min+max: 11 | đúng: 10


**Vì sao sai:** dãy con phải **giữ thứ tự** và mỗi phần tử dùng **một lần**. Ví dụ 1: chỉ lấy max cho 5 (thiếu); `max − min + max = 8` dùng `5` hai lần (thừa). Thứ tự đỉnh/đáy trong mảng quyết định.

### Lỗi 4 — Công thức tham lam quên phần tử đầu

In [24]:
def wrong_greedy_no_first(nums):
    return sum(max(0, nums[i] - nums[i - 1]) for i in range(1, len(nums)))   # SAI: thiếu nums[0]

for nums, want in examples:
    print(nums, "sai:", wrong_greedy_no_first(nums), "| đúng:", want)

[4, 2, 5, 3] sai: 3 | đúng: 7
[5, 6, 7, 8] sai: 3 | đúng: 8
[6, 2, 1, 2, 4, 5] sai: 4 | đúng: 10


**Vì sao sai:** khác bài cổ phiếu, ở đây phần tử đầu tiên của dãy con được **cộng nguyên** (như "mua ở giá 0"). Tương đương thêm `0` ảo trước `nums[0]`. **Sửa:** `ans = nums[0] + Σ …`.

### Lỗi 5 — Đệ quy có ghi nhớ trên `n = 10⁵`

In [25]:
long_nums = [rng.randint(1, 9) for _ in range(5000)]
try:
    max_alt_memo(long_nums)
    print("không lỗi")
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "| DP vòng lặp:", Solution().maxAlternatingSum(long_nums))

RecursionError: maximum recursion depth exceeded | DP vòng lặp: 7370


**Vì sao sai:** độ sâu đệ quy `f(i) → f(i+1)` bằng `n`, vượt giới hạn 1000. **Sửa:** DP vòng lặp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Dãy con với dấu xen kẽ / trạng thái luân phiên"** ⇒ DP hai trạng thái (chẵn/lẻ, cầm/không cầm…), giống họ bài **mua bán cổ phiếu**.
- Gán song song `a, b = …, …` để hai trạng thái dùng giá trị cũ.
- Khi DP có dạng `max(giữ, chuyển)` và các phần tử dương, thường có **tham lam theo đoạn tăng/giảm**.
- Phân biệt **dãy con** và **mảng con** ngay khi đọc đề.
- `n = 10⁵` ⇒ tránh đệ quy sâu.

## 15. Biến thể của bài

1. **Mảng con liên tiếp** có tổng xen kẽ lớn nhất ⇒ Kadane hai trạng thái (Bài tập 5).
2. **LeetCode 122** — mua bán cổ phiếu nhiều lần: cùng công thức "tổng bước tăng" (không cộng `nums[0]`).
3. **Có phí mỗi lần trừ** (LeetCode 714 tương tự) ⇒ DP hai trạng thái với `odd − x − fee`.
4. **Giới hạn số phần tử tối đa `2k + 1`** ⇒ DP `O(n·k)` (như LeetCode 188).
5. **Có truy vấn cập nhật phần tử** ⇒ segment tree lưu ma trận 2×2 trạng thái (max-plus).

### Minh họa biến thể 5: cập nhật phần tử + truy vấn đáp án bằng segment tree (max-plus)

Mỗi phần tử `x` là một phép biến đổi trạng thái `(even, odd) → (max(even, odd − x), max(odd, even + x))`. Viết dưới dạng **ma trận 2×2 trong đại số max-plus** (phép "nhân" là `+`, phép "cộng" là `max`):

```
[even']   [ 0   −x ] ⊗ [even]
[odd' ] = [ +x   0 ]   [odd ]
```

Ghép các phần tử liên tiếp = nhân ma trận max-plus (có tính kết hợp) ⇒ segment tree lưu tích ma trận của mỗi đoạn; cập nhật một phần tử `O(log n)`. (Ví dụ tự dựng — đối chiếu bằng chạy lại DP sau mỗi cập nhật.)

In [26]:
NEG = -inf

def mat_of(x):
    return ((0, -x), (x, 0))

def mat_mul(A, B):        # (A ⊗ B)[i][j] = max_k A[i][k] + B[k][j]; áp dụng B trước rồi A
    return tuple(tuple(max(A[i][k] + B[k][j] for k in range(2)) for j in range(2)) for i in range(2))

class AltSumTree:
    def __init__(self, nums):
        self.n = len(nums); self.size = 1
        while self.size < self.n: self.size *= 2
        I = ((0, NEG), (NEG, 0))                  # ma trận đơn vị max-plus
        self.t = [I] * (2 * self.size)
        for i, x in enumerate(nums): self.t[self.size + i] = mat_of(x)
        for i in range(self.size - 1, 0, -1): self.t[i] = mat_mul(self.t[2 * i + 1], self.t[2 * i])

    def update(self, i, x):
        i += self.size; self.t[i] = mat_of(x); i //= 2
        while i:
            self.t[i] = mat_mul(self.t[2 * i + 1], self.t[2 * i]); i //= 2

    def answer(self):
        M = self.t[1]
        even, odd = 0, NEG                         # trạng thái đầu
        return max(M[1][0] + even, M[1][1] + odd)  # odd' cuối cùng

rng_t = random.Random(23)
arr = [rng_t.randint(1, 30) for _ in range(17)]
tree = AltSumTree(arr)
assert tree.answer() == Solution().maxAlternatingSum(arr)
for _ in range(300):
    i, v = rng_t.randrange(len(arr)), rng_t.randint(1, 30)
    arr[i] = v; tree.update(i, v)
    assert tree.answer() == Solution().maxAlternatingSum(arr)
print("segment tree max-plus khớp DP sau 300 cập nhật; đáp án cuối =", tree.answer())

segment tree max-plus khớp DP sau 300 cập nhật; đáp án cuối = 91


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi dãy con | `O(2ⁿ n)` | `O(n)` | kiểm chứng |
| 2. DP kết thúc tại `i` | `E[i]`, `O[i]` | `O(n²)` | `O(n)` | bước trung gian |
| 3. DP hai trạng thái | `even`, `odd` | `O(n)` | `O(1)` | **nộp bài**, tổng quát hóa tốt |
| 4. Tham lam | tổng bước tăng + `nums[0]` | `O(n)` | `O(1)` | ngắn nhất, cần chứng minh |
| 5. Đệ quy nhớ | `f(i, dấu)` | `O(n)` | `O(n)` | dễ nghĩ, nhưng vỡ stack với `n` lớn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[1]`, `[3,1,3]`, `[2,3,1,4]`?

**Bài 2 (tính tay):** Lập bảng `even/odd` cho Ví dụ 1 `[4,2,5,3]`.

**Bài 3 (điền TODO):** Viết `best_indices(nums)` trả về **chỉ số** của một dãy con tối ưu bằng cách **truy vết DP** (lưu nguồn gốc của `even`/`odd`).

**Bài 4 (tự viết lại):** Viết lời giải bằng một vòng `for` dùng `zip(nums, nums[1:])` theo công thức tham lam.

**Bài 5 (mini challenge):** Tổng xen kẽ lớn nhất của một **mảng con liên tiếp** (khác rỗng), `O(n)`.

In [27]:
def best_indices(nums):
    # TODO
    pass

def max_alt_zip(nums):
    # TODO
    pass

def max_alt_subarray(nums):
    # TODO: Kadane hai trạng thái
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Một phần tử: chính nó | `3 − 1 + 3 = 5` | `[2,3,1,4]`: `3 − 1 + 4 = 6` |
| 2 | `(0, −∞)` | sau 4: `(0, 4)`, sau 2: `(2, 4)` | sau 5: `(2, 7)`, sau 3: `(4, 7)` |
| 3 | Lưu `(giá trị, danh sách chỉ số)` cho mỗi trạng thái | Khi chọn `x`, nối chỉ số vào danh sách của trạng thái nguồn | Để `O(n)` bộ nhớ: lưu con trỏ cha thay vì danh sách |
| 4 | `ans = nums[0]` | `for a, b in zip(nums, nums[1:])` | `ans += max(0, b − a)` |
| 5 | `pos` = tốt nhất kết thúc tại `i`, `nums[i]` được cộng | `neg` = kết thúc tại `i`, `nums[i]` bị trừ | `pos = max(x, neg + x)`, `neg = pos_cũ − x` |

## 19. Đáp án tham khảo

In [28]:
r1 = [Solution().maxAlternatingSum(a) for a in ([1], [3, 1, 3], [2, 3, 1, 4])]
print("Bài 1:", r1); assert r1 == [1, 5, 6]
print("Bài 2:"); assert trace_dp([4, 2, 5, 3]) == 7

Bài 1: [1, 5, 6]
Bài 2:
x=4: odd-x=-inf, even+x=4 -> even=0, odd=4
x=2: odd-x=2, even+x=2 -> even=2, odd=4
x=5: odd-x=-1, even+x=7 -> even=2, odd=7
x=3: odd-x=4, even+x=5 -> even=4, odd=7


In [29]:
def best_indices(nums):
    # trạng thái: (giá trị, chỉ số cuối, con trỏ cha) — cha trỏ vào node trước đó
    nodes = []                                    # node: (chỉ số trong nums, node cha)
    even = (0, -1)                                # (giá trị, node id; -1 = dãy rỗng)
    odd = (-inf, None)
    for i, x in enumerate(nums):
        new_even, new_odd = even, odd
        if odd[0] - x > even[0]:
            nodes.append((i, odd[1])); new_even = (odd[0] - x, len(nodes) - 1)
        if even[0] + x > odd[0]:
            nodes.append((i, even[1])); new_odd = (even[0] + x, len(nodes) - 1)
        even, odd = new_even, new_odd
    out, k = [], odd[1]
    while k is not None and k != -1:
        i, k = nodes[k]; out.append(i)
    return out[::-1]

def max_alt_zip(nums):
    ans = nums[0]
    for a, b in zip(nums, nums[1:]):
        ans += max(0, b - a)
    return ans

rng = random.Random(3)
for _ in range(800):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 12))]
    idx = best_indices(nums)
    assert idx == sorted(idx) and len(set(idx)) == len(idx)
    assert alt_sum([nums[i] for i in idx]) == max_alt_brute(nums) == max_alt_zip(nums)
print("Bài 3, 4: OK", best_indices([6, 2, 1, 2, 4, 5]))

Bài 3, 4: OK [0, 2, 5]


In [30]:
def max_alt_subarray(nums):
    pos, neg, best = -inf, -inf, -inf
    for x in nums:
        pos, neg = max(x, neg + x), pos - x
        best = max(best, pos, neg)
    return best

def max_alt_subarray_brute(nums):
    return max(alt_sum(nums[l:r + 1]) for l in range(len(nums)) for r in range(l, len(nums)))

rng = random.Random(8)
for _ in range(800):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 12))]
    assert max_alt_subarray(nums) == max_alt_subarray_brute(nums), nums
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", [max_alt_subarray(n) for n, _ in examples])

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK [7, 8, 7]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
even, odd = 0, -inf                 # even: chẵn phần tử (kế tiếp +), odd: lẻ (kế tiếp −)
for x in nums:
    even, odd = max(even, odd - x), max(odd, even + x)   # GÁN SONG SONG
return odd

# tham lam tương đương:
ans = nums[0] + sum(max(0, b - a) for a, b in zip(nums, nums[1:]))
```

**Checklist:**
- [ ] Đây là **dãy con** (được bỏ phần tử), không phải mảng con?
- [ ] Khởi tạo `even = 0`, `odd = -inf`?
- [ ] Gán song song (hai trạng thái dùng giá trị cũ)?
- [ ] Trả về `odd` (kết thúc bằng phép cộng)?
- [ ] Không đệ quy sâu `n`; Python không tràn số (ngôn ngữ khác dùng 64-bit).